# Эксперимент 9: Нормализация по маршруту

**Гипотеза:** разброс масштабов маршрутов (50K–981K) мешает модели — дерево разбивает пространство
одинаково для маршрута-50К и маршрута-981К, хотя их паттерны по форме одинаковы, а по масштабу — нет.
Если привести target и лаговые фичи к единому масштабу (делением на `route_mean`),
модель сможет лучше выучить форму паттернов, а не их абсолютный уровень.

**Реализация:**
- `target_norm = target / route_mean` — обучаем на безразмерном таргете (~1.0)
- Все lag / rolling / ewm / diff фичи таргета тоже делим на `route_mean`
- `route_mean` остаётся в фичах как контекст — модель может использовать абсолютный масштаб
- После инференса: обратная нормализация `pred = pred_norm × route_mean`
- Status-фичи **не** нормируются (у них своя природа, не связанная с route_mean)

**Структура:** 8 отдельных моделей (как в Эксп. 4), изолируем только эффект нормализации.

**Валидация:** rolling-origin, 3 субботних cutoff.
Метрика: **WAPE + |Relative Bias|**. Бейзлайн: **0.382**.

In [7]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import time

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 5)
sns.set_theme(style='whitegrid', palette='muted')

DATA_DIR = Path('/Users/melikhovartem/Desktop/ИЗИ 200к/Data')

STEP     = pd.Timedelta('30min')
HORIZONS = list(range(1, 9))

VAL_CUTOFFS = [
    pd.Timestamp('2025-10-11 10:30:00'),
    pd.Timestamp('2025-10-18 10:30:00'),
    pd.Timestamp('2025-10-25 10:30:00'),
]
TEST_CUTOFF = pd.Timestamp('2025-11-01 10:30:00')

SUBSAMPLE_N = 4

# Те же гиперпараметры, что в Эксп. 4 — изолируем эффект нормализации
_BASE = dict(
    objective='regression_l1',
    max_depth=8, num_leaves=63, min_child_samples=50,
    subsample=0.8, colsample_bytree=0.8,
    n_jobs=-1, random_state=42, verbose=-1,
)
H_PARAMS = {
    1: dict(**_BASE, n_estimators=1500, learning_rate=0.03, reg_alpha=0.05, reg_lambda=0.5),
    2: dict(**_BASE, n_estimators=1000, learning_rate=0.05, reg_alpha=0.1,  reg_lambda=1.0),
    3: dict(**_BASE, n_estimators=1000, learning_rate=0.05, reg_alpha=0.1,  reg_lambda=1.0),
    4: dict(**_BASE, n_estimators=1000, learning_rate=0.05, reg_alpha=0.1,  reg_lambda=1.0),
    5: dict(**_BASE, n_estimators=1000, learning_rate=0.05, reg_alpha=0.3,  reg_lambda=2.0),
    6: dict(**_BASE, n_estimators=1000, learning_rate=0.05, reg_alpha=0.3,  reg_lambda=2.0),
    7: dict(**_BASE, n_estimators=800,  learning_rate=0.05, reg_alpha=0.5,  reg_lambda=3.0),
    8: dict(**_BASE, n_estimators=800,  learning_rate=0.05, reg_alpha=0.5,  reg_lambda=3.0),
}

# Результат Эксп. 4 для сравнения (заполни после запуска Эксп. 4)
EXP4_TOTAL = 0.3269   # например: EXP4_TOTAL = 0.3269

print("Конфигурация загружена.")

Конфигурация загружена.


In [8]:
train = pd.read_parquet(DATA_DIR / 'train_solo_track.parquet')
test  = pd.read_parquet(DATA_DIR / 'test_solo_track.parquet')

train['timestamp'] = pd.to_datetime(train['timestamp'])
test['timestamp']  = pd.to_datetime(test['timestamp'])

train = train.sort_values(['route_id', 'timestamp']).reset_index(drop=True)
test  = test.sort_values(['route_id', 'timestamp']).reset_index(drop=True)

print(f"Train: {train.shape}  [{train.timestamp.min()} … {train.timestamp.max()}]")
print(f"Test:  {test.shape}   [{test.timestamp.min()} … {test.timestamp.max()}]")

Train: (4630000, 9)  [2025-07-28 00:00:00 … 2025-11-01 10:30:00]
Test:  (8000, 3)   [2025-11-01 11:00:00 … 2025-11-01 14:30:00]


In [9]:
def wape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.sum(np.abs(y_true - y_pred)) / np.sum(y_true))

def relative_bias(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float((np.sum(y_pred) - np.sum(y_true)) / np.sum(y_true))

def combined_metric(y_true, y_pred):
    return wape(y_true, y_pred) + abs(relative_bias(y_true, y_pred))

BASELINE = 0.382
print(f"Метрики определены. Бейзлайн: {BASELINE}")

Метрики определены. Бейзлайн: 0.382


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# Feature Engineering (идентично Эксп. 4)
# ══════════════════════════════════════════════════════════════════════════════

def compute_route_stats(data):
    stats = (
        data.groupby('route_id')['target_1h']
        .agg(
            route_mean='mean', route_median='median', route_std='std',
            route_q25=lambda x: x.quantile(0.25),
            route_q75=lambda x: x.quantile(0.75),
            route_zero_frac=lambda x: (x == 0).mean(),
        )
        .reset_index()
    )
    stats['route_cv'] = stats['route_std'] / (stats['route_mean'] + 1.0)
    return stats


def compute_ref_features(data, route_stats):
    data = data.sort_values(['route_id', 'timestamp']).copy()

    for lag in [1, 2, 3, 4, 6, 12, 24, 48, 96, 336]:
        data[f'target_lag_{lag}'] = data.groupby('route_id')['target_1h'].shift(lag)

    for window, name in [(4,'2h'), (12,'6h'), (24,'12h'), (48,'24h'), (336,'7d')]:
        data[f'target_roll_mean_{name}'] = data.groupby('route_id')['target_1h'].transform(
            lambda x, w=window: x.shift(1).rolling(w, min_periods=1).mean()
        )

    for window, name in [(12,'6h'), (48,'24h')]:
        data[f'target_roll_std_{name}'] = data.groupby('route_id')['target_1h'].transform(
            lambda x, w=window: x.shift(1).rolling(w, min_periods=2).std()
        )

    data['target_ewm_span4'] = data.groupby('route_id')['target_1h'].transform(
        lambda x: x.shift(1).ewm(span=4, min_periods=1).mean()
    )
    data['target_diff_1']  = data.groupby('route_id')['target_1h'].diff(1)
    data['target_diff_48'] = data.groupby('route_id')['target_1h'].diff(48)

    for s in ['status_1', 'status_2', 'status_3', 'status_4', 'status_5']:
        data[f'{s}_at_ref'] = data[s]

    for lag in [1, 2, 3, 4]:
        data[f'status_3_lag_{lag}'] = data.groupby('route_id')['status_3'].shift(lag)

    for s in ['status_1', 'status_2', 'status_3']:
        data[f'{s}_roll_mean_2h'] = data.groupby('route_id')[s].transform(
            lambda x: x.shift(1).rolling(4, min_periods=1).mean()
        )
    for s in ['status_4', 'status_5']:
        data[f'{s}_roll_mean_24h'] = data.groupby('route_id')[s].transform(
            lambda x: x.shift(1).rolling(48, min_periods=1).mean()
        )

    data['ref_hour'] = data['timestamp'].dt.hour
    data['ref_dow']  = data['timestamp'].dt.dayofweek
    data = data.merge(route_stats, on='route_id', how='left')
    return data


def _add_tgt_time_features(df, h):
    df = df.copy()
    df['tgt_ts']  = df['timestamp'] + h * STEP
    tgt_hour = df['tgt_ts'].dt.hour + df['tgt_ts'].dt.minute / 60
    tgt_dow  = df['tgt_ts'].dt.dayofweek
    df['tgt_hour_float'] = tgt_hour
    df['tgt_dow']        = tgt_dow
    df['tgt_hour_sin']   = np.sin(2 * np.pi * tgt_hour / 24)
    df['tgt_hour_cos']   = np.cos(2 * np.pi * tgt_hour / 24)
    df['tgt_dow_sin']    = np.sin(2 * np.pi * tgt_dow / 7)
    df['tgt_dow_cos']    = np.cos(2 * np.pi * tgt_dow / 7)
    return df


_TGT_COLS = ['route_id', 'timestamp', 'target_1h']

def build_train_matrix_h(feat_df, train_full, h, subsample_n=4):
    feat_df = feat_df.sort_values(['route_id', 'timestamp']).copy()
    feat_df['_rn'] = feat_df.groupby('route_id').cumcount()
    feat_sub = feat_df[feat_df['_rn'] % subsample_n == 0].drop('_rn', axis=1).copy()
    feat_sub = _add_tgt_time_features(feat_sub, h)
    tgt_lookup = train_full[_TGT_COLS].rename(
        columns={'timestamp': 'tgt_ts', 'target_1h': 'target'})
    feat_sub = feat_sub.merge(tgt_lookup, on=['route_id', 'tgt_ts'], how='left')
    return feat_sub.dropna(subset=['target'])


def build_inference_h(feat_df, cutoff, h):
    cutoff_df = feat_df[feat_df['timestamp'] == cutoff].copy()
    assert len(cutoff_df) == 1000
    cutoff_df = _add_tgt_time_features(cutoff_df, h)
    return cutoff_df.sort_values('route_id').reset_index(drop=True)


print("Feature engineering определён.")

Feature engineering определён.


In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# Нормализация (ключевое новшество Эксп. 9)
# ══════════════════════════════════════════════════════════════════════════════

# Колонки, которые нормируем делением на route_mean
_NORM_COLS = (
    [f'target_lag_{lag}' for lag in [1,2,3,4,6,12,24,48,96,336]]
    + [f'target_roll_mean_{n}' for n in ['2h','6h','12h','24h','7d']]
    + [f'target_roll_std_{n}'  for n in ['6h','24h']]
    + ['target_ewm_span4', 'target_diff_1', 'target_diff_48']
)

def normalize_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Нормализовать target-фичи делением на route_mean.
    Возвращает копию — исходный DataFrame не изменяется.
    """
    df = df.copy()
    rm = df['route_mean'].values
    for col in _NORM_COLS:
        if col in df.columns:
            df[col] = df[col] / rm
    return df


# Список признаков — тот же, что в Эксп. 4 (нормализация не меняет набор колонок)
FEATURE_COLS = [
    'route_id',
    'route_mean', 'route_median', 'route_std',
    'route_q25', 'route_q75', 'route_zero_frac', 'route_cv',
    'tgt_hour_float', 'tgt_dow',
    'tgt_hour_sin', 'tgt_hour_cos', 'tgt_dow_sin', 'tgt_dow_cos',
    'ref_hour', 'ref_dow',
    'target_lag_1', 'target_lag_2', 'target_lag_3', 'target_lag_4',
    'target_lag_6', 'target_lag_12', 'target_lag_24',
    'target_lag_48', 'target_lag_96', 'target_lag_336',
    'target_roll_mean_2h', 'target_roll_mean_6h', 'target_roll_mean_12h',
    'target_roll_mean_24h', 'target_roll_mean_7d',
    'target_roll_std_6h', 'target_roll_std_24h',
    'target_ewm_span4', 'target_diff_1', 'target_diff_48',
    'status_1_at_ref', 'status_2_at_ref', 'status_3_at_ref',
    'status_4_at_ref', 'status_5_at_ref',
    'status_3_lag_1', 'status_3_lag_2', 'status_3_lag_3', 'status_3_lag_4',
    'status_1_roll_mean_2h', 'status_2_roll_mean_2h', 'status_3_roll_mean_2h',
    'status_4_roll_mean_24h', 'status_5_roll_mean_24h',
]
CAT_FEATURES = ['route_id']

print(f"Нормализуемых колонок: {len(_NORM_COLS)}")
print(f"Всего признаков: {len(FEATURE_COLS)}")
print(f"\nПосле normalize_df значения lag/rolling ≈ 1.0 (безразмерные),")
print(f"route_mean в фичах остаётся — модель знает абсолютный масштаб.")

Нормализуемых колонок: 20
Всего признаков: 50

После normalize_df значения lag/rolling ≈ 1.0 (безразмерные),
route_mean в фичах остаётся — модель знает абсолютный масштаб.


## Кросс-валидация

Пайплайн в каждом фолде:
1. `compute_ref_features` → сырые признаки
2. `normalize_df` → делим lag/rolling на `route_mean` (фичи)
3. Для каждого h: `target_norm = target / route_mean` (обучающий таргет)
4. После инференса: `pred = pred_norm × route_mean` → исходный масштаб → метрика

In [12]:
fold_results    = []
all_horizon_acc = {h: {'WAPE': [], 'Total': []} for h in HORIZONS}
last_models     = {}

for fold_idx, cutoff in enumerate(VAL_CUTOFFS):
    t0 = time.time()
    print(f"\n{'='*60}")
    print(f"Фолд {fold_idx+1}/{len(VAL_CUTOFFS)}: cutoff = {cutoff}")

    # ── 1. Сырые признаки ──────────────────────────────────────────────────
    train_cut   = train[train['timestamp'] <= cutoff].copy()
    route_stats = compute_route_stats(train_cut)
    print(f"  ref_features ({len(train_cut):,} строк) ...", end=' ', flush=True)
    feat_raw = compute_ref_features(train_cut, route_stats)
    print("готово")

    # ── 2. Нормализованные признаки ────────────────────────────────────────
    feat_norm = normalize_df(feat_raw)

    h_preds   = {}
    h_actuals = {}

    for h in HORIZONS:
        # Обучающая матрица (из нормализованных признаков)
        tr_h = build_train_matrix_h(feat_norm, train, h, subsample_n=SUBSAMPLE_N)

        # Нормализованный таргет
        tr_h['target_norm'] = tr_h['target'] / tr_h['route_mean']

        model_h = lgb.LGBMRegressor(**H_PARAMS[h])
        model_h.fit(
            tr_h[FEATURE_COLS], tr_h['target_norm'],
            categorical_feature=CAT_FEATURES,
        )

        # Инференс → денормализация → исходный масштаб
        inf_h      = build_inference_h(feat_norm, cutoff, h)
        pred_norm  = np.maximum(model_h.predict(inf_h[FEATURE_COLS]), 0)
        pred_h     = pred_norm * inf_h['route_mean'].values

        h_preds[h] = (inf_h['route_id'].values, pred_h)

        # Реальные значения
        tgt_ts = cutoff + h * STEP
        act_h  = (
            train[train['timestamp'] == tgt_ts][['route_id', 'target_1h']]
            .sort_values('route_id').reset_index(drop=True)
        )
        h_actuals[h] = act_h['target_1h'].values

        if fold_idx == len(VAL_CUTOFFS) - 1:
            last_models[h] = model_h

        print(f"  h={h} обучено", end='\r')

    print()

    # ── Агрегация ──────────────────────────────────────────────────────────
    preds_all   = np.concatenate([h_preds[h][1]   for h in HORIZONS])
    actuals_all = np.concatenate([h_actuals[h]     for h in HORIZONS])

    ratio     = actuals_all.sum() / preds_all.sum()
    preds_cal = preds_all * ratio

    wape_v  = wape(actuals_all, preds_cal)
    rbias_v = relative_bias(actuals_all, preds_cal)
    total_v = wape_v + abs(rbias_v)

    horizon_metrics = []
    offset = 0
    for h in HORIZONS:
        n   = len(h_actuals[h])
        y_h = actuals_all[offset:offset+n]
        p_h = preds_cal[offset:offset+n]
        w_h = wape(y_h, p_h)
        b_h = relative_bias(y_h, p_h)
        t_h = w_h + abs(b_h)
        horizon_metrics.append({'horizon': h, 'WAPE': w_h, 'RBias': b_h, 'Total': t_h})
        all_horizon_acc[h]['WAPE'].append(w_h)
        all_horizon_acc[h]['Total'].append(t_h)
        offset += n

    elapsed = time.time() - t0
    print(f"  WAPE={wape_v:.4f}  |RBias|={abs(rbias_v):.4f}  Total={total_v:.4f}")
    print(f"  ratio={ratio:.4f}  время={elapsed:.1f}с")

    fold_results.append({
        'fold': fold_idx + 1, 'cutoff': str(cutoff.date()),
        'WAPE': wape_v, 'RBias': rbias_v, 'Total': total_v,
        'ratio': ratio, 'horizon_metrics': horizon_metrics,
        'elapsed': elapsed,
        # Сохраняем для анализа по квартилям маршрутов
        'h_preds': h_preds, 'h_actuals': h_actuals, 'route_stats': route_stats,
    })

print(f"\n{'='*60}")
print("Валидация завершена.")


Фолд 1/3: cutoff = 2025-10-11 10:30:00
  ref_features (3,622,000 строк) ... готово
  h=8 обучено
  WAPE=0.3358  |RBias|=0.0000  Total=0.3358
  ratio=1.0246  время=282.8с

Фолд 2/3: cutoff = 2025-10-18 10:30:00
  ref_features (3,958,000 строк) ... готово
  h=8 обучено
  WAPE=0.3229  |RBias|=0.0000  Total=0.3229
  ratio=1.0545  время=338.5с

Фолд 3/3: cutoff = 2025-10-25 10:30:00
  ref_features (4,294,000 строк) ... готово
  h=8 обучено
  WAPE=0.3213  |RBias|=0.0000  Total=0.3213
  ratio=1.0393  время=400.3с

Валидация завершена.


In [13]:
rows = [
    {'Фолд': r['fold'], 'Cutoff': r['cutoff'],
     'WAPE': r['WAPE'], '|RBias|': abs(r['RBias']),
     'Total': r['Total'], 'Ratio': r['ratio'], 'Время, с': round(r['elapsed'])}
    for r in fold_results
]
res_df = pd.DataFrame(rows)
mean_row = res_df[['WAPE','|RBias|','Total','Ratio','Время, с']].mean().to_dict()
mean_row.update({'Фолд': 'Среднее', 'Cutoff': ''})
res_df = pd.concat([res_df, pd.DataFrame([mean_row])], ignore_index=True)

print("Результаты кросс-валидации:")
print(res_df.to_string(index=False, float_format='{:.4f}'.format))

mean_total = float(res_df.loc[res_df['Фолд'] == 'Среднее', 'Total'].values[0])
print(f"\nБейзлайн: {BASELINE:.4f}  →  Эксп 9: {mean_total:.4f}  (Δ = {mean_total-BASELINE:+.4f})")
if EXP4_TOTAL is not None:
    print(f"Эксп 4:   {EXP4_TOTAL:.4f}  →  Эксп 9: {mean_total:.4f}  (Δ = {mean_total-EXP4_TOTAL:+.4f})")

Результаты кросс-валидации:
   Фолд     Cutoff   WAPE  |RBias|  Total  Ratio  Время, с
      1 2025-10-11 0.3358   0.0000 0.3358 1.0246  283.0000
      2 2025-10-18 0.3229   0.0000 0.3229 1.0545  338.0000
      3 2025-10-25 0.3213   0.0000 0.3213 1.0393  400.0000
Среднее            0.3267   0.0000 0.3267 1.0395  340.3333

Бейзлайн: 0.3820  →  Эксп 9: 0.3267  (Δ = -0.0553)
Эксп 4:   0.3269  →  Эксп 9: 0.3267  (Δ = -0.0002)


In [ ]:
# Метрика по горизонтам
h_wape  = [np.mean(all_horizon_acc[h]['WAPE'])  for h in HORIZONS]
h_total = [np.mean(all_horizon_acc[h]['Total']) for h in HORIZONS]
h_labels = [f'+{h*30}м' for h in HORIZONS]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
x = np.arange(len(HORIZONS))
ax.bar(x - 0.2, h_total, 0.35, label='Total', color='steelblue', alpha=0.85)
ax.bar(x + 0.2, h_wape,  0.35, label='WAPE',  color='dodgerblue', alpha=0.85)
ax.axhline(BASELINE, color='red', linestyle='--', linewidth=1.5, label=f'Бейзлайн')
ax.axhline(mean_total, color='navy', linestyle=':', linewidth=1.5, label=f'Среднее ({mean_total:.4f})')
ax.set_xticks(x); ax.set_xticklabels(h_labels)
ax.set_title('Метрика по горизонтам (Эксп. 9)', fontsize=12)
ax.set_ylabel('Метрика'); ax.legend()

ax = axes[1]
for r in fold_results:
    ax.plot(HORIZONS, [hm['Total'] for hm in r['horizon_metrics']],
            'o-', alpha=0.7, label=f"Фолд {r['fold']} ({r['cutoff']})")
ax.plot(HORIZONS, h_total, 'k^--', linewidth=2, markersize=8, label='Среднее')
ax.axhline(BASELINE, color='red', linestyle='--', linewidth=1.5)
ax.set_xticks(HORIZONS); ax.set_xticklabels(h_labels)
ax.set_title('Total по горизонтам (все фолды)', fontsize=12)
ax.set_ylabel('WAPE + |RBias|'); ax.legend(fontsize=9)

plt.tight_layout(); plt.show()

In [ ]:
# ── Анализ по квартилям route_mean ────────────────────────────────────────
# Ключевой вопрос: помогает ли нормализация малым маршрутам?
# Используем последний фолд

last_fold = fold_results[-1]
rs = last_fold['route_stats'].copy()
rs['q_label'] = pd.qcut(rs['route_mean'], q=4,
                         labels=['Q1 (малые)', 'Q2', 'Q3', 'Q4 (большие)'])

quartile_results = {q: {'y': [], 'p': []} for q in rs['q_label'].cat.categories}

for h in HORIZONS:
    route_ids, preds_h = last_fold['h_preds'][h]
    actuals_h          = last_fold['h_actuals'][h]

    # Калибровочный ratio из этого фолда
    ratio = last_fold['ratio']
    preds_cal_h = preds_h * ratio

    route_q = rs.set_index('route_id')['q_label']
    for rid, pred, actual in zip(route_ids, preds_cal_h, actuals_h):
        q = route_q.get(rid)
        if q is not None:
            quartile_results[q]['y'].append(actual)
            quartile_results[q]['p'].append(pred)

q_labels, q_wapes, q_totals = [], [], []
for q, data in quartile_results.items():
    if data['y']:
        w = wape(data['y'], data['p'])
        t = combined_metric(data['y'], data['p'])
        q_labels.append(str(q))
        q_wapes.append(w)
        q_totals.append(t)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(q_labels))
ax.bar(x - 0.2, q_totals, 0.35, label='Total', color='steelblue', alpha=0.85)
ax.bar(x + 0.2, q_wapes,  0.35, label='WAPE',  color='dodgerblue', alpha=0.85)
ax.axhline(BASELINE, color='red', linestyle='--', linewidth=1.5, label=f'Бейзлайн')
ax.set_xticks(x); ax.set_xticklabels(q_labels)
ax.set_title('Метрика по квартилям route_mean (последний фолд)\n'
             'Ожидаем улучшение у Q1 (малые маршруты)', fontsize=12)
ax.set_ylabel('Метрика'); ax.legend()
plt.tight_layout(); plt.show()

print("WAPE по квартилям:")
for q, w, t in zip(q_labels, q_wapes, q_totals):
    print(f"  {q:20s}  WAPE={w:.4f}  Total={t:.4f}")

## Финальная модель + Тестовые прогнозы

In [14]:
print("Обучаю финальные модели на полном train ...")
t0 = time.time()

route_stats_full = compute_route_stats(train)
print("  ref_features (полный train) ...", end=' ', flush=True)
feat_raw_full  = compute_ref_features(train, route_stats_full)
feat_norm_full = normalize_df(feat_raw_full)
print("готово")

final_models = {}
for h in HORIZONS:
    tr_h = build_train_matrix_h(feat_norm_full, train, h, subsample_n=SUBSAMPLE_N)
    tr_h['target_norm'] = tr_h['target'] / tr_h['route_mean']
    m = lgb.LGBMRegressor(**H_PARAMS[h])
    m.fit(tr_h[FEATURE_COLS], tr_h['target_norm'], categorical_feature=CAT_FEATURES)
    final_models[h] = m
    print(f"  h={h} обучено", end='\r')

print(f"\nГотово ({time.time()-t0:.1f}с)")

mean_ratio = np.mean([r['ratio'] for r in fold_results])
print(f"Калибровочный коэф: {mean_ratio:.4f}")

sub_rows = []
for h in HORIZONS:
    inf_h      = build_inference_h(feat_norm_full, TEST_CUTOFF, h)
    pred_norm  = np.maximum(final_models[h].predict(inf_h[FEATURE_COLS]), 0)
    pred_h     = pred_norm * inf_h['route_mean'].values * mean_ratio
    tmp = inf_h[['route_id']].copy()
    tmp['timestamp'] = TEST_CUTOFF + h * STEP
    tmp['y_pred']    = pred_h   # float, без округления
    sub_rows.append(tmp)

sub = pd.concat(sub_rows, ignore_index=True)
test_ids = test[['id', 'route_id', 'timestamp']].copy()
sub = sub.merge(test_ids, on=['route_id', 'timestamp'], how='left')
sub = sub[['id', 'y_pred']].sort_values('id').reset_index(drop=True)

assert len(sub) == len(test)
assert sub['id'].notna().all()

out_path = '/Users/melikhovartem/Desktop/ИЗИ 200к/submission_exp09.csv'
sub.to_csv(out_path, index=False)
print(f"Submission сохранён: {out_path}")
print(f"Прогнозы: min={sub['y_pred'].min():.0f}  max={sub['y_pred'].max():.0f}  mean={sub['y_pred'].mean():.0f}")

Обучаю финальные модели на полном train ...
  ref_features (полный train) ... готово
  h=8 обучено
Готово (437.1с)
Калибровочный коэф: 1.0395
Submission сохранён: /Users/melikhovartem/Desktop/ИЗИ 200к/submission_exp09.csv
Прогнозы: min=0  max=1930611  mean=274751


## Итоговые метрики (для сравнения экспериментов)

In [ ]:
mean_wape  = np.mean([r['WAPE']       for r in fold_results])
mean_rbias = np.mean([abs(r['RBias']) for r in fold_results])
mean_total = np.mean([r['Total']      for r in fold_results])

print("=" * 65)
print("ИТОГОВЫЕ МЕТРИКИ — ЭКСПЕРИМЕНТ 9")
print("LightGBM × 8 + нормализация по route_mean")
print("=" * 65)
print(f"\nВалидация: rolling-origin, 3 субботних фолда\n")

summary_df = pd.DataFrame({
    'Метрика':              ['WAPE', '|Relative Bias|', 'Total (WAPE+|RBias|)'],
    'Среднее по 3 фолдам': [f'{mean_wape:.4f}', f'{mean_rbias:.4f}', f'{mean_total:.4f}'],
})
print(summary_df.to_string(index=False))

print(f"\n{'─'*65}")
rows_cmp = [
    {'Эксперимент': 'Exp 0: среднее по маршруту',         'Total': 0.382},
    {'Эксперимент': 'Exp 4: LightGBM × 8 (без норм.)',    'Total': EXP4_TOTAL if EXP4_TOTAL else float('nan')},
    {'Эксперимент': '▶ Exp 9: LightGBM × 8 + норм.',      'Total': mean_total},
]
print(pd.DataFrame(rows_cmp).to_string(index=False, float_format='{:.4f}'.format))

# ── Итоговый график ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
fold_labels = [r['cutoff'] for r in fold_results]
fold_totals = [r['Total']  for r in fold_results]
bars = ax.bar(fold_labels, fold_totals, color='steelblue', alpha=0.85, width=0.5)
ax.axhline(mean_total, color='navy', linewidth=2, label=f'Среднее = {mean_total:.4f}')
ax.axhline(BASELINE,   color='red',  linewidth=1.5, linestyle='--', label=f'Baseline = {BASELINE}')
for bar, v in zip(bars, fold_totals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.002, f'{v:.4f}',
            ha='center', va='bottom', fontsize=10)
ax.set_title('Total metric по фолдам', fontsize=12)
ax.set_ylabel('WAPE + |RBias|'); ax.legend()
ax.set_ylim(0, max(fold_totals) * 1.15)

ax = axes[1]
ax.plot(HORIZONS, h_total, 'o-', color='steelblue', linewidth=2, markersize=7, label='Total (Exp 9)')
ax.plot(HORIZONS, h_wape,  's--', color='dodgerblue', linewidth=1.5, markersize=6, label='WAPE (Exp 9)')
ax.axhline(BASELINE,   color='red',  linestyle='--', linewidth=1.5, label=f'Baseline')
ax.axhline(mean_total, color='navy', linestyle=':', linewidth=1.5, label=f'Среднее = {mean_total:.4f}')
ax.set_xticks(HORIZONS); ax.set_xticklabels([f'+{h*30}м' for h in HORIZONS])
ax.set_title('Total по горизонтам прогноза', fontsize=12)
ax.set_ylabel('Метрика'); ax.set_xlabel('Горизонт'); ax.legend(fontsize=9)

plt.suptitle('Эксперимент 9: LightGBM + нормализация по маршруту', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

print(f"\n{'='*65}")
print(f"  ИТОГ: Total = {mean_total:.4f}  (бейзлайн: {BASELINE}  Δ={mean_total-BASELINE:+.4f})")
print(f"{'='*65}")